# C11-neural-training — Session 1: Stable Softmax and Cross-Entropy

*One 90-minute session. Prerequisites: C5's output layer and shapes, F4's
partial derivatives and chain rule, and F1's broadcasting and aggregation
axes.*

**Learning contract.** For batch size $N$ and class count $C$, logits
$Z\in\mathbb R^{N\times C}$ become probabilities
$P\in\mathbb R^{N\times C}$. We will define every symbol, preserve this
shape, compute without overflow, and finish with both the softmax Jacobian and
the fused cross-entropy gradient.


In [ ]:
import numpy as np

SEED = 20260804
ATOL = 1e-10
RTOL = 1e-10
rng = np.random.default_rng(SEED)

## 1. Logits, probabilities, and the class axis

For example $i$ and class $c$, the model emits a **logit** $Z_{ic}$: an
unrestricted score, not a probability. Softmax converts the $C$ scores in one
row into

$$P_{ic}=\frac{e^{Z_{ic}}}{\sum_{j=1}^{C}e^{Z_{ij}}}.$$

Here $j$ is a dummy class index. Every $P_{ic}>0$ and each row sums to one.
The normalization axis is therefore class axis 1, never the batch axis 0.

For $Z=(\log 2,0)$, exponentials are $(2,1)$ and
$P=(2/3,1/3)$. For a batch, perform that same calculation independently in
each row.

**Checkpoint 1A.** What are the softmax probabilities of $(0,0,0)$?

**Checkpoint 1B.** If `Z.shape == (4, 3)`, state the shapes of the row
denominator and $P$ when the denominator keeps its dimension.


In [ ]:
Z_small = np.array([[np.log(2.0), 0.0], [0.0, 0.0]])
raw = np.exp(Z_small)
denominator = raw.sum(axis=1, keepdims=True)
P_small = raw / denominator
print("logits", Z_small.shape, "denominator", denominator.shape, "probabilities", P_small.shape)
print(P_small)
assert np.allclose(P_small.sum(axis=1), 1.0, atol=ATOL, rtol=RTOL)

## 2. Shift invariance gives a stable algorithm

For any scalar $a$ added to every logit in one example,

$$\frac{e^{Z_c+a}}{\sum_j e^{Z_j+a}}
=\frac{e^a e^{Z_c}}{e^a\sum_j e^{Z_j}}
=\frac{e^{Z_c}}{\sum_j e^{Z_j}}.$$

Thus softmax is **shift invariant**. Choose $a=-m$, where
$m=\max_j Z_j$. Every shifted logit $Z_j-m\le 0$, so every exponential lies
in $(0,1]$ and cannot overflow. The stable batch algorithm keeps
$m\in\mathbb R^{N\times1}$ for broadcasting.

**Checkpoint 2A.** Why can subtracting one global maximum for the whole batch
be numerically unsafe even though it is algebraically valid for every row?

**Checkpoint 2B.** After rowwise shifting, what is the largest exponential in
each row?


In [ ]:
def stable_softmax(logits):
    """Rowwise softmax for logits of shape (N, C)."""
    logits = np.asarray(logits, dtype=float)
    row_max = logits.max(axis=1, keepdims=True)          # (N, 1)
    shifted = logits - row_max                          # (N, C)
    exp_shifted = np.exp(shifted)                       # (N, C)
    return exp_shifted / exp_shifted.sum(axis=1, keepdims=True)

huge = np.array([[1000.0, 1001.0, 999.0], [-1000.0, -999.0, -1002.0]])
P_huge = stable_softmax(huge)
P_shifted = stable_softmax(huge + np.array([[37.0], [-91.0]]))
print(P_huge)
assert np.all(np.isfinite(P_huge))
assert np.allclose(P_huge, P_shifted, atol=ATOL, rtol=RTOL)
assert np.allclose(P_huge.sum(axis=1), 1.0, atol=ATOL, rtol=RTOL)

## 3. Cross-entropy is negative log likelihood

Let $y_i\in\{0,\ldots,C-1\}$ be the correct class index for example $i$.
Its one-hot vector $Y_i$ has $Y_{ic}=1$ only when $c=y_i$.
Categorical cross-entropy is

$$L=-\frac1N\sum_{i=1}^{N}\sum_{c=1}^{C}Y_{ic}\log P_{ic}
=-\frac1N\sum_{i=1}^{N}\log P_{i,y_i}.$$

The second form is the **negative log likelihood**: penalize the log
probability assigned to the observed class. The mean makes the batch gradient
an average. A confident correct prediction has loss near zero; assigning zero
probability would have infinite loss.

A stable log probability avoids computing a tiny $P$ and then taking its log:

$$\log P_{ic}=(Z_{ic}-m_i)-\log\sum_j e^{Z_{ij}-m_i}.$$

This is the log-sum-exp construction. We neither clip probabilities nor add an
arbitrary epsilon: those operations change the stated mathematical loss.

**Checkpoint 3A.** For probabilities $(2/3,1/3)$ and target class 1 (zero
based), what is the loss?

**Checkpoint 3B.** If two per-example losses are $0.2$ and $1.0$, what is the
mean loss and why does its gradient include $1/2$?


In [ ]:
def stable_cross_entropy(logits, labels):
    """Mean categorical CE for logits (N, C), integer labels (N,)."""
    logits = np.asarray(logits, dtype=float)
    labels = np.asarray(labels, dtype=int)
    n = logits.shape[0]
    shifted = logits - logits.max(axis=1, keepdims=True)
    log_normalizer = np.log(np.exp(shifted).sum(axis=1))  # (N,)
    correct_shifted = shifted[np.arange(n), labels]        # (N,)
    losses = log_normalizer - correct_shifted              # (N,)
    return losses.mean()

loss = stable_cross_entropy(np.array([[np.log(2.0), 0.0]]), np.array([1]))
print("loss =", loss, "expected log(3) =", np.log(3.0))
assert np.isclose(loss, np.log(3.0), atol=ATOL, rtol=RTOL)

## 4. Worked exam-register example 1: stable normal form

**Problem.** One example has logits $(1000+\log2,1000)$ and correct class
index 1. Let $g=\partial L/\partial Z_1$, the derivative with respect to the
*first* logit. If $g=a/b$ in lowest terms with $b>0$, report $a+b$.
Reasoning is required; direct evaluation of `exp(1000)` is forbidden.

**Solution.**

1. Subtract the maximum $1000+\log2$. The logits become
   $(0,-\log2)$.
2. Exponentials are $(1,1/2)$, so probabilities are $(2/3,1/3)$.
3. For softmax plus cross-entropy, derived fully in Section 6,
   $\nabla_Z L=P-Y$. The target class is the second class, so
   $Y=(0,1)$.
4. Therefore $\nabla_ZL=(2/3,-2/3)$ and $g=2/3$.
5. The fraction is reduced, $a=2$, $b=3$, hence **$a+b=5$**.

The shift was not cosmetic: the naive exponentials overflow while the stable
derivation stays exact.

**Checkpoint 4A.** Repeat for target class 0. What is the first gradient
component?

**Checkpoint 4B.** Why must the denominator sign convention $b>0$ be stated
in a normal-form item?


In [ ]:
exam_logits = np.array([[1000.0 + np.log(2.0), 1000.0]])
exam_p = stable_softmax(exam_logits)
exam_y = np.array([[0.0, 1.0]])
exam_g = exam_p - exam_y
print("P =", exam_p, "gradient =", exam_g)
assert np.allclose(exam_p, [[2 / 3, 1 / 3]], atol=ATOL, rtol=RTOL)
assert np.allclose(exam_g, [[2 / 3, -2 / 3]], atol=ATOL, rtol=RTOL)

## 5. The softmax Jacobian

For one example, write $p_c=e^{z_c}/S$ with
$S=\sum_j e^{z_j}$. The Jacobian entry
$J_{ck}=\partial p_c/\partial z_k$ has two cases.

- If $c=k$, the numerator and denominator depend on $z_c$:
  $\partial p_c/\partial z_c=p_c(1-p_c)$.
- If $c\ne k$, only the denominator depends on $z_k$:
  $\partial p_c/\partial z_k=-p_cp_k$.

Using the Kronecker delta $\delta_{ck}$, which is 1 when $c=k$ and 0
otherwise,

$$J_{ck}=p_c(\delta_{ck}-p_k),\qquad
J=\operatorname{diag}(p)-pp^\top.$$

Thus $J\in\mathbb R^{C\times C}$. Every row and column sums to zero,
reflecting shift invariance: moving logits together changes no probability.

**Checkpoint 5A.** Write the $2\times2$ Jacobian for $p=(2/3,1/3)$.

**Checkpoint 5B.** Why does $J\mathbf 1=0$ encode shift invariance?


In [ ]:
p = np.array([2 / 3, 1 / 3])
J = np.diag(p) - np.outer(p, p)
expected_J = np.array([[2 / 9, -2 / 9], [-2 / 9, 2 / 9]])
print(J)
assert np.allclose(J, expected_J, atol=ATOL, rtol=RTOL)
assert np.allclose(J.sum(axis=0), 0.0, atol=ATOL, rtol=RTOL)
assert np.allclose(J.sum(axis=1), 0.0, atol=ATOL, rtol=RTOL)

## 6. Fusing softmax and cross-entropy

For one example, $L=-\sum_c y_c\log p_c$. First,
$\partial L/\partial p_c=-y_c/p_c$. Chain through every probability:

$$\frac{\partial L}{\partial z_k}
=\sum_c\frac{-y_c}{p_c}p_c(\delta_{ck}-p_k)
=-y_k+p_k\sum_c y_c=p_k-y_k,$$

because one-hot labels satisfy $\sum_c y_c=1$.
For a mean over $N$ examples,

$$\boxed{\frac{\partial L}{\partial Z}=\frac{P-Y}{N}}.$$

Both sides have shape $(N,C)$. Each gradient row sums to zero. The fusion is
more than shorter algebra: implementations can obtain the stable gradient
without materializing a full $C\times C$ Jacobian for each example.

**Checkpoint 6A.** For $N=4$, what factor distinguishes a mean-loss gradient
from a sum-loss gradient?

**Checkpoint 6B.** If one gradient row does not sum to zero, name two likely
bugs.


In [ ]:
def fused_ce_gradient(logits, labels):
    logits = np.asarray(logits, dtype=float)
    labels = np.asarray(labels, dtype=int)
    n = logits.shape[0]
    gradient = stable_softmax(logits)
    gradient[np.arange(n), labels] -= 1.0
    return gradient / n

Z = np.array([[2.0, -1.0, 0.5], [-0.5, 0.25, 1.25]])
y = np.array([0, 2])
G = fused_ce_gradient(Z, y)
print("gradient shape:", G.shape, "\n", G)
assert G.shape == Z.shape
assert np.allclose(G.sum(axis=1), 0.0, atol=ATOL, rtol=RTOL)

## 7. Common pitfalls and exam connections

**Broken example 1 — naive overflow.** `np.exp(logits)` can become infinity.
Subtract the row maximum before exponentiating.

**Broken example 2 — wrong axis.** `sum(axis=0)` normalizes across examples,
coupling unrelated rows. Use `axis=1, keepdims=True`.

**Broken example 3 — unstable loss.** `-np.log(softmax(...)[target])` can log
an underflowed zero. Compute the selected shifted logit and log-sum-exp
directly.

**Broken example 4 — missing mean factor.** $P-Y$ is the gradient of a *sum*
over examples. Divide by $N$ when the reported loss is the mean.

Round 1 questions commonly hide these contracts inside a short code trace:
an axis choice, a huge logit, a target-index gather, or a gradient normal
form. State shapes before arithmetic and identify whether the loss is summed
or averaged.

**Going deeper.** C11 Session 2 sends this $(N,C)$ gradient backward through
the MLP. C7 keeps the same loss while replacing dense feature extraction with
convolution.

**Checkpoint 7A.** Diagnose a function that returns rows summing to $1/N$.

**Checkpoint 7B.** Is adding $10^{-9}$ to every softmax denominator the same
mathematical function? Explain.


In [ ]:
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(np.array([[1000.0, 1001.0]]))
    naive = naive / naive.sum(axis=1, keepdims=True)
stable = stable_softmax(np.array([[1000.0, 1001.0]]))
print("naive:", naive, "| stable:", stable)
assert not np.all(np.isfinite(naive))
assert np.all(np.isfinite(stable))

## Checkpoint answers

**1A.** $(1/3,1/3,1/3)$. **1B.** Denominator $(4,1)$; $P$ $(4,3)$.

**2A.** Subtracting one global scalar preserves every row's exact softmax
ratios. Numerically, however, a lower-scale row may have all exponentials
underflow to zero, producing a zero denominator and $0/0$. Subtracting each
row's own maximum guarantees that row contains an exponential equal to 1.
**2B.** Exactly $1$, attained by every maximum logit.

**3A.** $-\log(1/3)=\log3$. **3B.** $0.6$; differentiating the mean multiplies
the sum of example gradients by $1/2$.

**4A.** $2/3-1=-1/3$. **4B.** Without $b>0$, $(a,b)$ and $(-a,-b)$ are both
reduced representations, so the requested integer is ambiguous.

**5A.** $\begin{pmatrix}2/9&-2/9\\-2/9&2/9\end{pmatrix}$.
**5B.** The directional derivative in the all-ones shift direction is zero.

**6A.** Divide by 4. **6B.** A missing one-hot subtraction or normalization
along the wrong axis (also possibly inconsistent averaging).

**7A.** It normalized probabilities by batch size after softmax; probabilities,
unlike mean gradients, must sum to one per row. **7B.** No. It changes every
probability and breaks exact normalization; stable shifting solves overflow
without changing the function.
